### Beyond the Squeaky Wheel: 311 Engagement & Equity Analysis
### Notebook 4: 311 Service Request Index Construction (311 SRI)

Transforms raw per-tract 311 point counts into the 311 Service Request Index (SRI). Applies log, square-root, and Yeo-Johnson transformations, rescales to a 0-100 scale per city, and exports comparison histograms alongside the final tract-level index.

In [ ]:
# Step 1: Import libraries

import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import PowerTransformer
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
# Step 2: Upload data and set file paths

TRACTS_PATH = "INSERT FILE PATH: tracts with 311 point counts GeoPackage"
tracts = gpd.read_file(TRACTS_PATH, engine="pyogrio")

PDF_OUTPUT_PATH = "INSERT FILE PATH: output histogram PDF"

cities = sorted(tracts["city"].dropna().unique())

In [ ]:
# Step 3: Apply transformations and min-max rescale

def min_max_scale(s):
    return (s - s.min()) / (s.max() - s.min()) * 100

# ---- TRANSFORMATIONS ----
tracts["log_point_count"] = np.log1p(tracts["point_count"])
tracts["sqrt_point_count"] = np.sqrt(tracts["point_count"])

# Yeo-Johnson fit per city; NaN (excluded outlier tracts) dropped before
# fitting, then reinserted at their original positions afterward
yj_values = pd.Series(index=tracts.index, dtype=float)
yj_lambdas = {}

for city, group in tracts.groupby("city"):
    valid = group["point_count"].dropna()
    pt = PowerTransformer(method="yeo-johnson")
    transformed = pt.fit_transform(valid.to_frame())
    yj_lambdas[city] = pt.lambdas_[0]
    yj_values.loc[valid.index] = transformed.flatten()

tracts["yj_point_count"] = yj_values

print("Yeo-Johnson lambda per city:")
for city, lam in yj_lambdas.items():
    print(f"  {city}: {lam:.4f}")

# ---- SCALE EACH TRANSFORMATION 0-100 WITHIN EACH CITY ----
tracts["311_score_raw"] = tracts.groupby("city")["point_count"].transform(min_max_scale).round(2)
tracts["311_score_log"] = tracts.groupby("city")["log_point_count"].transform(min_max_scale).round(2)
tracts["311_score_sqrt"] = tracts.groupby("city")["sqrt_point_count"].transform(min_max_scale).round(2)
tracts["311_score_yj"] = tracts.groupby("city")["yj_point_count"].transform(min_max_scale).round(2)

In [ ]:
# Step 4: Generate histograms for each city and export PDF

# ---- HISTOGRAMS: 2x2 per city, one page per city ----
cities = sorted(tracts["city"].dropna().unique())

transform_cols = [
    ("point_count", "raw"),
    ("log_point_count", "log1p"),
    ("sqrt_point_count", "sqrt"),
    ("yj_point_count", "yeo-johnson")
]

with PdfPages(PDF_OUTPUT_PATH) as pdf:
    for city in cities:
        subset = tracts[tracts["city"] == city]

        fig, axes = plt.subplots(2, 2, figsize=(10, 8))
        axes = axes.flatten()

        for ax, (col, label) in zip(axes, transform_cols):
            ax.hist(subset[col].dropna(), bins=30, color="steelblue", edgecolor="white")
            ax.set_title(f"{city} - {label}")

        plt.suptitle(city, fontsize=14)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.show()

print(f"Saved comparison histograms to {PDF_OUTPUT_PATH}")

In [ ]:
# Step 5: Export final tract-level GeoPackage and CSV

FINAL_TRACTS_OUTPUT_PATH = "INSERT FILE PATH: final tract-level SRI output"

tracts.to_file(FINAL_TRACTS_OUTPUT_PATH, driver="GPKG", engine="pyogrio")
tracts.drop(columns="geometry").to_csv(FINAL_TRACTS_OUTPUT_PATH.replace(".gpkg", ".csv"), index=False)

print(f"Saved GeoPackage and CSV to {FINAL_TRACTS_OUTPUT_PATH}")